[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/doxav/astromodel_proving/blob/main/analysis/07_assumption_sensitivity.ipynb)

# Step 07 - Assumption sensitivity: gating, proxy, and compartment split

This notebook runs the local Step 07 pipeline without Google Drive dependencies and writes auditable outputs under `outputs/assumption_sensitivity/`.

**Scope:** reviewer-facing assumption checks use all accepted cells by default with one best accepted candidate per cell. This keeps the assumption screen tied to the full cell-level validation scope without arbitrary row caps.

**Claim scope:** Step 07 audits whether Step 04-06 conclusions are robust to modeling assumptions. It does not authorize final biological degeneracy wording because parameter plausibility and synthesis checks remain pending.

In [ ]:
from pathlib import Path
import os
import pandas as pd
import matplotlib.pyplot as plt

PROJECT_ROOT = Path(os.environ.get("ASTROMODEL_PROJECT_ROOT", ".")).resolve()
PROJECT_ROOT

In [ ]:
from src.step07_assumption_sensitivity import Step07Config, run_step07_assumption_sensitivity

config = Step07Config(
    max_candidates=None,
    candidate_policy="best_per_cell",
    time_points=40,
    write_outputs=True,
)
result = run_step07_assumption_sensitivity(PROJECT_ROOT, config)
result["analysis_summary"]

## Accepted ensemble inventory

The Step 07 input contract preserves cell identity, region, condition, candidate ID, and mechanism labels from upstream steps.

In [ ]:
gating = result["gating_family_comparison"]
inventory = gating[["file_id", "region", "condition", "candidate_id", "mechanism_cluster", "dominant_mechanism"]].drop_duplicates()
inventory.head(12)

## Gating-family comparison

All configured gating families are scored using the same candidate/current/time-grid contract.

In [ ]:
gating[["gating_family", "region", "condition", "candidate_id", "current_na", "simulation_status", "trace_rmse_vs_sigmoid_mV", "family_supported_under_same_contract", "mechanism_claim_stable"]].head(18)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
plot_df = gating[gating["simulation_status"].eq("ok")].copy()
if not plot_df.empty:
    plot_df.boxplot(column="trace_rmse_vs_sigmoid_mV", by="gating_family", ax=ax, rot=30)
    ax.set_ylabel("RMSE versus sigmoid baseline (mV)")
    ax.set_title("Gating-family divergence under identical contract")
    fig.suptitle("")
else:
    ax.text(0.5, 0.5, "No successful gating simulations", ha="center")
plt.tight_layout()

## Model-comparison summary

In [ ]:
model = result["model_comparison"]
model

## Intracellular K proxy validity

The local intracellular proxy `ΔK_a,t` is compared with simulated extracellular `K_o` using correlation, scaled RMSE, and lag metrics. Rows that fail criteria explicitly require an ECS variant or additional data.

In [ ]:
proxy = result["proxy_validity_by_ensemble"]
proxy[["file_id", "region", "condition", "candidate_id", "current_na", "pearson_r", "spearman_r", "scaled_rmse", "best_lag_samples", "proxy_validity_status", "explicit_ecs_variant_required"]].head(12)

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
if not proxy.empty:
    ax.scatter(proxy["pearson_r"], proxy["scaled_rmse"], c=proxy["explicit_ecs_variant_required"].astype(int), cmap="coolwarm", s=80)
    ax.axvline(config.proxy_corr_min, color="black", linestyle="--", linewidth=1, label="corr threshold")
    ax.axhline(config.proxy_rmse_max, color="gray", linestyle=":", linewidth=1, label="RMSE threshold")
    ax.set_xlabel("Pearson r: ΔK_a,t vs K_o")
    ax.set_ylabel("Scaled RMSE after linear rescaling")
    ax.legend()
else:
    ax.text(0.5, 0.5, "No proxy rows", ha="center")
plt.tight_layout()

## Compartment-split sensitivity

The two-state local proxy is compared with a one-state aggregate proxy. This is a sensitivity score rather than a replacement model fit.

In [ ]:
split = result["compartment_split_sensitivity"]
split[["file_id", "region", "condition", "candidate_id", "current_na", "two_state_proxy_abs_corr", "one_state_proxy_abs_corr", "corr_delta_one_minus_two", "split_sensitivity_status", "mechanism_structure_persists"]].head(12)

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
if not split.empty:
    ax.scatter(split["two_state_proxy_abs_corr"], split["one_state_proxy_abs_corr"], s=80)
    ax.plot([0, 1], [0, 1], color="black", linestyle="--", linewidth=1)
    ax.set_xlabel("Two-state |corr(ΔK_a,t, K_o)|")
    ax.set_ylabel("One-state |corr(ΔK_a,total, K_o)|")
    ax.set_xlim(0, 1.05)
    ax.set_ylim(0, 1.05)
else:
    ax.text(0.5, 0.5, "No compartment rows", ha="center")
plt.tight_layout()

## Conservative claim scope

Final biological degeneracy language remains disallowed after Step 07. The table below separates robust, model-dependent, and unresolved assumption axes.

In [ ]:
claims = result["claim_scope_table"]
claims

In [ ]:
output_dir = PROJECT_ROOT / "outputs" / "assumption_sensitivity"
print("Wrote:")
for path in sorted(output_dir.glob("*")):
    print("-", path.relative_to(PROJECT_ROOT))

In [ ]:
assert result["analysis_summary"]["n_candidates"] >= 1
assert not result["claim_scope_table"]["final_degeneracy_claim_allowed_after_step07"].astype(bool).any()
print("Step 07 full-cell assumption sensitivity notebook completed across the full cell-level target scope.")

## Post-execution scientific status

Executed status for reviewer response: Step 07 evaluated 30 candidates across gating, proxy, and compartment-split assumption screens. The local/syncytial compartment split is `split_robust`, but gating remains `model_dependent_or_insufficient_evidence` and the intracellular-K proxy still requires explicit ECS-variant or extra-data support. This directly answers R3 with an objective limitation rather than an overclaim; final degeneracy wording remains disallowed after Step 07.